In [1]:
from collections import deque


# ============================================================
# 1. Surface-code stabilizers
# ============================================================

x_stabilizers = [
    [0, 1],
    [1, 2, 3, 4],
    [4, 5, 6, 7],
    [7, 8],
]

z_stabilizers = [
    [5, 6],
    [0, 1, 4, 5],
    [3, 4, 7, 8],
    [2, 3],
]


# ============================================================
# 2. X-error decoding graph
#
# Each data qubit is an edge.
# Its endpoints are the Z stabilizers affected by an X error.
# A qubit belonging to only one Z stabilizer connects to a
# boundary.
# ============================================================

x_error_graph = {
    0: ("Z1", "B_top"),
    1: ("Z1", "B_top"),
    2: ("Z3", "B_top"),
    3: ("Z2", "Z3"),
    4: ("Z1", "Z2"),
    5: ("Z0", "Z1"),
    6: ("Z0", "B_bottom"),
    7: ("Z2", "B_bottom"),
    8: ("Z2", "B_bottom"),
}


# ============================================================
# 3. Graph in adjacency-list form
#
# Format:
#     node -> [(neighbor, physical_data_qubit), ...]
# ============================================================

graph = {
    "Z0": [
        ("Z1", 5),
        ("B_bottom", 6),
    ],

    "Z1": [
        ("B_top", 0),
        ("B_top", 1),
        ("Z2", 4),
        ("Z0", 5),
    ],

    "Z2": [
        ("Z3", 3),
        ("Z1", 4),
        ("B_bottom", 7),
        ("B_bottom", 8),
    ],

    "Z3": [
        ("B_top", 2),
        ("Z2", 3),
    ],

    "B_top": [
        ("Z1", 0),
        ("Z1", 1),
        ("Z3", 2),
    ],

    "B_bottom": [
        ("Z0", 6),
        ("Z2", 7),
        ("Z2", 8),
    ],
}


# ============================================================
# 4. Calculate syndrome from physical X errors
#
# Each affected stabilizer is toggled.
# Two errors affecting the same stabilizer cancel:
#
#     (-1) * (-1) = +1
#
# This is equivalent to addition modulo 2.
# ============================================================

def syndrome_from_x_errors(errors, graph):

    defects = set()

    for q in errors:

        a, b = x_error_graph[q]

        for node in (a, b):

            # Boundaries are not syndrome defects
            if node.startswith("Z"):

                if node in defects:
                    defects.remove(node)
                else:
                    defects.add(node)

    return sorted(defects)


# ============================================================
# 5. Convert Qiskit's displayed syndrome string to defects
#
# Qiskit displays classical bits in reverse order.
#
# Example:
#
#     "0011" -> classical bits [1,1,0,0]
#             -> defects Z0, Z1
# ============================================================

def syndrome_to_defects(syndrome):

    bits = syndrome[::-1]

    return [
        i
        for i, bit in enumerate(bits)
        if bit == "1"
    ]


# ============================================================
# 6. Shortest path
#
# All physical-error edges currently have weight = 1,
# so Breadth-First Search is sufficient.
#
# Returns:
#     physical data-qubit indices
#     path length
# ============================================================

def shortest_path(graph, start, goal):

    queue = deque([
        (start, [], 0)
    ])

    visited = {start}

    while queue:

        node, path, distance = queue.popleft()

        if node == goal:
            return path, distance

        for neighbor, qubit in graph[node]:

            if neighbor not in visited:

                visited.add(neighbor)

                queue.append(
                    (
                        neighbor,
                        path + [qubit],
                        distance + 1
                    )
                )

    return None, float("inf")


# ============================================================
# 7. Generate all perfect matchings
#
# Example:
#
#     [Z0, Z1, Z2, Z3]
#
# produces:
#
#     (Z0,Z1) + (Z2,Z3)
#     (Z0,Z2) + (Z1,Z3)
#     (Z0,Z3) + (Z1,Z2)
# ============================================================

def all_matchings(nodes):

    if len(nodes) == 0:
        yield []
        return

    first = nodes[0]

    for i in range(1, len(nodes)):

        second = nodes[i]

        remaining = nodes[1:i] + nodes[i+1:]

        for matching in all_matchings(remaining):

            yield [(first, second)] + matching


# ============================================================
# 8. MWPM
#
# This version handles defect-to-defect matching only.
# Boundary matching will be added later.
# ============================================================

def mwpm(defects, graph):

    best_matching = None
    best_weight = float("inf")

    for matching in all_matchings(defects):

        total_weight = 0

        valid = True

        for a, b in matching:

            _, weight = shortest_path(
                graph,
                a,
                b
            )

            if weight == float("inf"):
                valid = False
                break

            total_weight += weight

        if valid and total_weight < best_weight:

            best_weight = total_weight
            best_matching = matching

    return best_matching, best_weight


# ============================================================
# 9. Full X-error decoder
#
# MWPM determines which defects should be paired.
# Shortest paths determine which physical qubits form
# the correction chains.
# ============================================================

def decode_x_errors(defects, graph):

    matching, weight = mwpm(
        defects,
        graph
    )

    correction = []

    for a, b in matching:

        path, path_weight = shortest_path(
            graph,
            a,
            b
        )

        correction.extend(path)

    return correction, matching, weight


# ============================================================
# 10. Example tests
# ============================================================

print("Single-qubit X errors:")
for q in range(9):

    defects = syndrome_from_x_errors(
        [q],
        x_error_graph
    )

    print(f"X{q} -> {defects}")


print("\nChain X5 X4:")

errors = [5, 4]

defects = syndrome_from_x_errors(
    errors,
    x_error_graph
)

print("Errors:   ", errors)
print("Syndrome: ", defects)


print("\nChain X5 X4 X3:")

errors = [5, 4, 3]

defects = syndrome_from_x_errors(
    errors,
    x_error_graph
)

print("Errors:   ", errors)
print("Syndrome: ", defects)


print("\nMWPM example:")

defects = ["Z0", "Z1", "Z2", "Z3"]

correction, matching, weight = decode_x_errors(
    defects,
    graph
)

print("Defects:    ", defects)
print("Matching:   ", matching)
print("Weight:     ", weight)
print("Correction: ", correction)

print(
    "Correction syndrome:",
    syndrome_from_x_errors(
        correction,
        x_error_graph
    )
)

Single-qubit X errors:
X0 -> ['Z1']
X1 -> ['Z1']
X2 -> ['Z3']
X3 -> ['Z2', 'Z3']
X4 -> ['Z1', 'Z2']
X5 -> ['Z0', 'Z1']
X6 -> ['Z0']
X7 -> ['Z2']
X8 -> ['Z2']

Chain X5 X4:
Errors:    [5, 4]
Syndrome:  ['Z0', 'Z2']

Chain X5 X4 X3:
Errors:    [5, 4, 3]
Syndrome:  ['Z0', 'Z3']

MWPM example:
Defects:     ['Z0', 'Z1', 'Z2', 'Z3']
Matching:    [('Z0', 'Z1'), ('Z2', 'Z3')]
Weight:      2
Correction:  [5, 3]
Correction syndrome: ['Z0', 'Z1', 'Z2', 'Z3']


In [2]:
from collections import deque


# ============================================================
# 1. Code definition
# ============================================================

z_stabilizers = [
    [5, 6],
    [0, 1, 4, 5],
    [3, 4, 7, 8],
    [2, 3],
]


# ============================================================
# 2. Build qubit -> stabilizer incidence
# ============================================================

def build_incidence(stabilizers):

    incidence = {}

    for check, qubits in enumerate(stabilizers):

        for q in qubits:
            incidence.setdefault(q, []).append(check)

    return incidence


z_incidence = build_incidence(z_stabilizers)


# ============================================================
# 3. Build interior decoder edges
#
# A data qubit belonging to two Z stabilizers is an edge
# connecting those two syndrome vertices.
# ============================================================

def build_decoder_edges(stabilizers):

    incidence = build_incidence(stabilizers)

    edges = {}
    boundary_qubits = {}

    for q, checks in incidence.items():

        if len(checks) == 2:

            a, b = checks

            edges[q] = (
                f"Z{a}",
                f"Z{b}"
            )

        elif len(checks) == 1:

            a = checks[0]

            boundary_qubits[q] = f"Z{a}"

        else:

            raise ValueError(
                f"Unexpected number of checks for q{q}: {checks}"
            )

    return edges, boundary_qubits


edges, boundary_qubits = build_decoder_edges(
    z_stabilizers
)


# ============================================================
# 4. Physical boundary definition
#
# This comes from the geometry of this particular d=3 code.
# ============================================================

boundary_connections = {

    "B_top": {
        "Z1": [0, 1],
        "Z3": [2],
    },

    "B_bottom": {
        "Z0": [6],
        "Z2": [7, 8],
    },
}


# ============================================================
# 5. Build stabilizer adjacency graph
#
# IMPORTANT:
# boundaries are NOT graph nodes here.
#
# They are handled separately.
# ============================================================

graph = {}

for check in range(len(z_stabilizers)):
    graph[f"Z{check}"] = []


for q, (a, b) in edges.items():

    graph[a].append((b, q))
    graph[b].append((a, q))


# ============================================================
# 6. Shortest distance between two stabilizers
# ============================================================

def stabilizer_distance(graph, start, goal):

    queue = deque([
        (start, 0)
    ])

    visited = {start}

    while queue:

        node, distance = queue.popleft()

        if node == goal:
            return distance

        for neighbor, qubit in graph[node]:

            if neighbor not in visited:

                visited.add(neighbor)

                queue.append(
                    (neighbor, distance + 1)
                )

    return float("inf")


# ============================================================
# 7. Distance from stabilizer to boundary
#
# We find the shortest path from the stabilizer to any
# physical data qubit touching that boundary.
#
# The final boundary edge itself contributes one unit.
# ============================================================

def boundary_distance(
    graph,
    boundary_connections,
    start,
    boundary
):

    if boundary not in boundary_connections:
        return float("inf")

    best = float("inf")

    for stabilizer, qubits in boundary_connections[boundary].items():

        distance = stabilizer_distance(
            graph,
            start,
            stabilizer
        )

        if distance == float("inf"):
            continue

        # The final physical qubit connects the stabilizer
        # to the boundary.
        distance += 1

        best = min(best, distance)

    return best


# ============================================================
# 8. Display graph
# ============================================================

print("Decoder graph:")
for node, neighbors in graph.items():
    print(node, "->", neighbors)


# ============================================================
# 9. Stabilizer-to-stabilizer distances
# ============================================================

print("\nStabilizer distances:")

for a in graph:

    for b in graph:

        if a < b:

            d = stabilizer_distance(
                graph,
                a,
                b
            )

            print(
                f"d({a}, {b}) = {d}"
            )


# ============================================================
# 10. Stabilizer-to-boundary distances
# ============================================================

print("\nBoundary distances:")

for stabilizer in graph:

    for boundary in boundary_connections:

        d = boundary_distance(
            graph,
            boundary_connections,
            stabilizer,
            boundary
        )

        print(
            f"d({stabilizer}, {boundary}) = {d}"
        )

Decoder graph:
Z0 -> [('Z1', 5)]
Z1 -> [('Z0', 5), ('Z2', 4)]
Z2 -> [('Z1', 4), ('Z3', 3)]
Z3 -> [('Z2', 3)]

Stabilizer distances:
d(Z0, Z1) = 1
d(Z0, Z2) = 2
d(Z0, Z3) = 3
d(Z1, Z2) = 1
d(Z1, Z3) = 2
d(Z2, Z3) = 1

Boundary distances:
d(Z0, B_top) = 2
d(Z0, B_bottom) = 1
d(Z1, B_top) = 1
d(Z1, B_bottom) = 2
d(Z2, B_top) = 2
d(Z2, B_bottom) = 1
d(Z3, B_top) = 1
d(Z3, B_bottom) = 2


In [3]:
def boundary_aware_matching(
    defects,
    graph,
    boundary_connections
):

    best = None
    best_weight = float("inf")

    def recurse(remaining, matching, weight):

        nonlocal best, best_weight

        # ----------------------------------------------------
        # Everything has been matched
        # ----------------------------------------------------

        if not remaining:

            if weight < best_weight:

                best_weight = weight
                best = matching.copy()

            return


        # ----------------------------------------------------
        # Prune expensive branches
        # ----------------------------------------------------

        if weight >= best_weight:
            return


        # ----------------------------------------------------
        # Take first remaining defect
        # ----------------------------------------------------

        defect = remaining[0]

        rest = remaining[1:]


        # ====================================================
        # Option 1:
        # Match defect to another defect
        # ====================================================

        for i, other in enumerate(rest):

            remaining_after = (
                rest[:i] +
                rest[i+1:]
            )

            distance = stabilizer_distance(
                graph,
                defect,
                other
            )

            if distance == float("inf"):
                continue

            matching.append(
                (defect, other, distance)
            )

            recurse(
                remaining_after,
                matching,
                weight + distance
            )

            matching.pop()


        # ====================================================
        # Option 2:
        # Match defect to a boundary
        # ====================================================

        for boundary in boundary_connections:

            distance = boundary_distance(
                graph,
                boundary_connections,
                defect,
                boundary
            )

            if distance == float("inf"):
                continue

            matching.append(
                (defect, boundary, distance)
            )

            recurse(
                rest,
                matching,
                weight + distance
            )

            matching.pop()


    recurse(
        defects,
        [],
        0
    )

    return best, best_weight

In [4]:
tests = [
    ["Z0"],
    ["Z1"],
    ["Z2"],
    ["Z3"],
    ["Z0", "Z1"],
    ["Z1", "Z2"],
    ["Z0", "Z2"],
    ["Z0", "Z1", "Z2"],
    ["Z0", "Z1", "Z2", "Z3"],
]


for defects in tests:

    matching, weight = boundary_aware_matching(
        defects,
        graph,
        boundary_connections
    )

    print()
    print("Defects: ", defects)
    print("Match:   ", matching)
    print("Weight:  ", weight)


Defects:  ['Z0']
Match:    [('Z0', 'B_bottom', 1)]
Weight:   1

Defects:  ['Z1']
Match:    [('Z1', 'B_top', 1)]
Weight:   1

Defects:  ['Z2']
Match:    [('Z2', 'B_bottom', 1)]
Weight:   1

Defects:  ['Z3']
Match:    [('Z3', 'B_top', 1)]
Weight:   1

Defects:  ['Z0', 'Z1']
Match:    [('Z0', 'Z1', 1)]
Weight:   1

Defects:  ['Z1', 'Z2']
Match:    [('Z1', 'Z2', 1)]
Weight:   1

Defects:  ['Z0', 'Z2']
Match:    [('Z0', 'Z2', 2)]
Weight:   2

Defects:  ['Z0', 'Z1', 'Z2']
Match:    [('Z0', 'Z1', 1), ('Z2', 'B_bottom', 1)]
Weight:   2

Defects:  ['Z0', 'Z1', 'Z2', 'Z3']
Match:    [('Z0', 'Z1', 1), ('Z2', 'Z3', 1)]
Weight:   2


In [5]:
from collections import deque


# ============================================================
# 1. Find shortest path between two stabilizers
#
# Returns:
#     path of physical data qubits
#     path length
# ============================================================

def shortest_path(graph, start, goal):

    queue = deque([
        (start, [], 0)
    ])

    visited = {start}

    while queue:

        node, path, distance = queue.popleft()

        if node == goal:
            return path, distance

        for neighbor, qubit in graph[node]:

            if neighbor not in visited:

                visited.add(neighbor)

                queue.append(
                    (
                        neighbor,
                        path + [qubit],
                        distance + 1
                    )
                )

    return None, float("inf")


# ============================================================
# 2. Find shortest path from a stabilizer to a boundary
#
# We explicitly choose the physical boundary qubit at the end.
# ============================================================

def shortest_path_to_boundary(
    graph,
    boundary_connections,
    start,
    boundary
):

    best_path = None
    best_distance = float("inf")

    for stabilizer, boundary_qubits in boundary_connections[boundary].items():

        # Path through the stabilizer graph
        path, distance = shortest_path(
            graph,
            start,
            stabilizer
        )

        if distance == float("inf"):
            continue

        # We must finally traverse one physical data qubit
        # from that stabilizer to the boundary.
        for q in boundary_qubits:

            candidate_path = path + [q]
            candidate_distance = distance + 1

            if candidate_distance < best_distance:

                best_path = candidate_path
                best_distance = candidate_distance

    return best_path, best_distance


# ============================================================
# 3. Convert one MWPM match into physical qubits
# ============================================================

def matching_to_path(
    match,
    graph,
    boundary_connections
):

    a, b, weight = match

    # --------------------------------------------------------
    # Defect <-> defect
    # --------------------------------------------------------

    if a.startswith("Z") and b.startswith("Z"):

        path, distance = shortest_path(
            graph,
            a,
            b
        )

        return path


    # --------------------------------------------------------
    # Defect <-> boundary
    # --------------------------------------------------------

    if a.startswith("Z") and b.startswith("B"):

        path, distance = shortest_path_to_boundary(
            graph,
            boundary_connections,
            a,
            b
        )

        return path


    # Boundary <-> defect
    # --------------------------------------------------------

    if a.startswith("B") and b.startswith("Z"):

        path, distance = shortest_path_to_boundary(
            graph,
            boundary_connections,
            b,
            a
        )

        return path


    raise ValueError(
        f"Invalid matching: {match}"
    )


# ============================================================
# 4. Convert complete matching into correction qubits
# ============================================================

def matching_to_correction(
    matching,
    graph,
    boundary_connections
):

    correction = []

    for match in matching:

        path = matching_to_path(
            match,
            graph,
            boundary_connections
        )

        correction.extend(path)

    # --------------------------------------------------------
    # Corrections are Pauli X operators.
    #
    # If the same qubit appears twice:
    #
    #       X * X = I
    #
    # so it cancels.
    # --------------------------------------------------------

    correction = sorted(
        set(correction)
    )

    return correction

In [6]:
tests = [

    # Z0 -> Z1
    [("Z0", "Z1", 1)],

    # Z1 -> top boundary
    [("Z1", "B_top", 1)],

    # Z0 -> bottom boundary
    [("Z0", "B_bottom", 1)],

    # Z0 -> Z2
    [("Z0", "Z2", 2)],

    # Two independent paths
    [
        ("Z0", "Z1", 1),
        ("Z2", "Z3", 1),
    ],
]


for matching in tests:

    correction = matching_to_correction(
        matching,
        graph,
        boundary_connections
    )

    print()
    print("Matching:   ", matching)
    print("Correction: ", correction)


Matching:    [('Z0', 'Z1', 1)]
Correction:  [5]

Matching:    [('Z1', 'B_top', 1)]
Correction:  [0]

Matching:    [('Z0', 'B_bottom', 1)]
Correction:  [6]

Matching:    [('Z0', 'Z2', 2)]
Correction:  [4, 5]

Matching:    [('Z0', 'Z1', 1), ('Z2', 'Z3', 1)]
Correction:  [3, 5]


In [7]:
z_stabilizers = [
    [5, 6],
    [0, 1, 4, 5],
    [3, 4, 7, 8],
    [2, 3],
]

In [8]:
def syndrome_to_defects(syndrome):
    """
    Qiskit displays classical bits in reverse order.

    Example:
        '0011' -> classical bits [1, 1, 0, 0]
                 -> defects Z0, Z1
    """
    bits = syndrome[::-1]
    return [
        i for i, bit in enumerate(bits)
        if bit == "1"
    ]

In [9]:
def error_syndrome(error_qubits, stabilizers):
    """
    Return stabilizer indices with odd overlap
    with the X-error chain.
    """
    syndrome = []

    for i, stabilizer in enumerate(stabilizers):
        overlap = len(set(error_qubits) & set(stabilizer))

        if overlap % 2 == 1:
            syndrome.append(i)

    return syndrome

In [10]:
for q in range(9):
    syndrome = error_syndrome([q], z_stabilizers)
    print(f"X{q} -> Z{syndrome}")

X0 -> Z[1]
X1 -> Z[1]
X2 -> Z[3]
X3 -> Z[2, 3]
X4 -> Z[1, 2]
X5 -> Z[0, 1]
X6 -> Z[0]
X7 -> Z[2]
X8 -> Z[2]


In [11]:
def decode_x_error(
    defects,
    graph,
    boundary_connections,
):
    matching, weight = boundary_aware_matching(
        [f"Z{i}" for i in defects],
        graph,
        boundary_connections,
    )

    if matching is None:
        return None, None, None

    correction = matching_to_correction(
        matching,
        graph,
        boundary_connections,
    )

    return matching, correction, weight

In [12]:
def verify_correction(
    original_defects,
    correction,
    stabilizers,
):
    correction_defects = error_syndrome(
        correction,
        stabilizers,
    )

    return set(original_defects) == set(correction_defects)

In [13]:
print("Single-X error decoding")
print("=" * 50)

for q in range(9):

    # Actual physical error
    actual_error = [q]

    # Syndrome
    defects = error_syndrome(
        actual_error,
        z_stabilizers,
    )

    # Decode
    matching, correction, weight = decode_x_error(
        defects,
        graph,
        boundary_connections,
    )

    # Verify syndrome
    success = verify_correction(
        defects,
        correction,
        z_stabilizers,
    )

    print()
    print(f"Actual error : X{q}")
    print(f"Syndrome     : Z{defects}")
    print(f"Matching     : {matching}")
    print(f"Correction   : X{correction}")
    print(f"Weight       : {weight}")
    print(f"Syndrome OK  : {success}")

Single-X error decoding

Actual error : X0
Syndrome     : Z[1]
Matching     : [('Z1', 'B_top', 1)]
Correction   : X[0]
Weight       : 1
Syndrome OK  : True

Actual error : X1
Syndrome     : Z[1]
Matching     : [('Z1', 'B_top', 1)]
Correction   : X[0]
Weight       : 1
Syndrome OK  : True

Actual error : X2
Syndrome     : Z[3]
Matching     : [('Z3', 'B_top', 1)]
Correction   : X[2]
Weight       : 1
Syndrome OK  : True

Actual error : X3
Syndrome     : Z[2, 3]
Matching     : [('Z2', 'Z3', 1)]
Correction   : X[3]
Weight       : 1
Syndrome OK  : True

Actual error : X4
Syndrome     : Z[1, 2]
Matching     : [('Z1', 'Z2', 1)]
Correction   : X[4]
Weight       : 1
Syndrome OK  : True

Actual error : X5
Syndrome     : Z[0, 1]
Matching     : [('Z0', 'Z1', 1)]
Correction   : X[5]
Weight       : 1
Syndrome OK  : True

Actual error : X6
Syndrome     : Z[0]
Matching     : [('Z0', 'B_bottom', 1)]
Correction   : X[6]
Weight       : 1
Syndrome OK  : True

Actual error : X7
Syndrome     : Z[2]
Matching  

In [14]:
logical_x = {0, 5, 6}

In [15]:
x_stabilizers = [
    {0, 1},
    {1, 2, 3, 4},
    {4, 5, 6, 7},
    {7, 8},
]

In [16]:
def symmetric_difference(a, b):
    return set(a) ^ set(b)


def is_stabilizer_x(error):
    """
    Check whether an X-type operator is in the
    X-stabilizer group.
    """
    error = set(error)

    generators = x_stabilizers

    # Enumerate all products of the 4 X stabilizers
    for mask in range(16):
        product = set()

        for i, stabilizer in enumerate(generators):
            if mask & (1 << i):
                product ^= stabilizer

        if product == error:
            return True

    return False


def is_logical_x(error):
    """
    Check whether error is equivalent to X_L
    up to an X-type stabilizer.
    """
    error = set(error)

    equivalent = error ^ logical_x

    return is_stabilizer_x(equivalent)

In [17]:
def residual_error(actual, correction):
    return set(actual) ^ set(correction)

In [18]:
from itertools import combinations

failures = []

for q1, q2 in combinations(range(9), 2):

    actual = {q1, q2}

    # Syndrome
    defects = error_syndrome(
        actual,
        z_stabilizers,
    )

    # Decode
    matching, correction, weight = decode_x_error(
        defects,
        graph,
        boundary_connections,
    )

    correction = set(correction)

    # Check syndrome
    correction_defects = set(
        error_syndrome(
            correction,
            z_stabilizers,
        )
    )

    syndrome_ok = (
        set(defects) == correction_defects
    )

    # Residual after correction
    residual = residual_error(
        actual,
        correction,
    )

    logical_error = is_logical_x(residual)

    if not syndrome_ok or logical_error:
        failures.append({
            "actual": actual,
            "defects": defects,
            "matching": matching,
            "correction": correction,
            "residual": residual,
            "syndrome_ok": syndrome_ok,
            "logical_error": logical_error,
        })

    print(
        f"X{q1}X{q2: <2} "
        f"syndrome={defects!s:10} "
        f"correction={sorted(correction)!s:10} "
        f"residual={sorted(residual)}"
    )

X0X1  syndrome=[]         correction=[]         residual=[0, 1]
X0X2  syndrome=[1, 3]     correction=[3, 4]     residual=[0, 2, 3, 4]
X0X3  syndrome=[1, 2, 3]  correction=[2, 4]     residual=[0, 2, 3, 4]
X0X4  syndrome=[2]        correction=[7]        residual=[0, 4, 7]
X0X5  syndrome=[0]        correction=[6]        residual=[0, 5, 6]
X0X6  syndrome=[0, 1]     correction=[5]        residual=[0, 5, 6]
X0X7  syndrome=[1, 2]     correction=[4]        residual=[0, 4, 7]
X0X8  syndrome=[1, 2]     correction=[4]        residual=[0, 4, 8]
X1X2  syndrome=[1, 3]     correction=[3, 4]     residual=[1, 2, 3, 4]
X1X3  syndrome=[1, 2, 3]  correction=[2, 4]     residual=[1, 2, 3, 4]
X1X4  syndrome=[2]        correction=[7]        residual=[1, 4, 7]
X1X5  syndrome=[0]        correction=[6]        residual=[1, 5, 6]
X1X6  syndrome=[0, 1]     correction=[5]        residual=[1, 5, 6]
X1X7  syndrome=[1, 2]     correction=[4]        residual=[1, 4, 7]
X1X8  syndrome=[1, 2]     correction=[4]        resid

In [19]:
def is_x_stabilizer(error):
    error = set(error)

    for mask in range(16):
        product = set()

        for i, stabilizer in enumerate(x_stabilizers):
            if mask & (1 << i):
                product ^= stabilizer

        if product == error:
            return True

    return False


def is_logical_x(error):
    error = set(error)

    # E is equivalent to X_L if E * X_L is a stabilizer
    return is_x_stabilizer(error ^ logical_x)

In [20]:
def classify_residual(residual):
    residual = set(residual)

    if is_x_stabilizer(residual):
        return "SUCCESS: stabilizer"

    if is_logical_x(residual):
        return "FAILURE: logical X"

    return "OTHER"

In [21]:
for q1, q2 in combinations(range(9), 2):

    actual = {q1, q2}

    defects = error_syndrome(
        actual,
        z_stabilizers,
    )

    matching, correction, weight = decode_x_error(
        defects,
        graph,
        boundary_connections,
    )

    correction = set(correction)

    residual = actual ^ correction

    syndrome_ok = (
        set(error_syndrome(
            correction,
            z_stabilizers,
        ))
        == set(defects)
    )

    classification = classify_residual(residual)

    print(
        f"X{q1}X{q2:<2} "
        f"syndrome={defects!s:<12} "
        f"correction={sorted(correction)!s:<10} "
        f"residual={sorted(residual)!s:<15} "
        f"{classification}"
    )

X0X1  syndrome=[]           correction=[]         residual=[0, 1]          SUCCESS: stabilizer
X0X2  syndrome=[1, 3]       correction=[3, 4]     residual=[0, 2, 3, 4]    SUCCESS: stabilizer
X0X3  syndrome=[1, 2, 3]    correction=[2, 4]     residual=[0, 2, 3, 4]    SUCCESS: stabilizer
X0X4  syndrome=[2]          correction=[7]        residual=[0, 4, 7]       FAILURE: logical X
X0X5  syndrome=[0]          correction=[6]        residual=[0, 5, 6]       FAILURE: logical X
X0X6  syndrome=[0, 1]       correction=[5]        residual=[0, 5, 6]       FAILURE: logical X
X0X7  syndrome=[1, 2]       correction=[4]        residual=[0, 4, 7]       FAILURE: logical X
X0X8  syndrome=[1, 2]       correction=[4]        residual=[0, 4, 8]       FAILURE: logical X
X1X2  syndrome=[1, 3]       correction=[3, 4]     residual=[1, 2, 3, 4]    SUCCESS: stabilizer
X1X3  syndrome=[1, 2, 3]    correction=[2, 4]     residual=[1, 2, 3, 4]    SUCCESS: stabilizer
X1X4  syndrome=[2]          correction=[7]        resid